# Myownai — أول تجربة تدريب على GPU مجاني

**مهم:** التجربة دي للتأكد إن الكود والـGPU والتدريب وحفظ الأوزان شغالين. هنستخدم نصوصًا تجريبية مصطنعة، لذلك النتيجة ليست مساعدًا ذكيًا ولن تعلّم الموديل العربية أو الإنجليزية بشكل حقيقي. بعد نجاح الاختبار هنربط مجموعة بيانات مرخّصة.

قبل التشغيل: من إعدادات الـNotebook في Kaggle اختار **Accelerator → GPU**، ثم شغّل الخلايا بالترتيب من أعلى لأسفل.


In [ ]:
import torch, os, sys, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('لم يتم تفعيل GPU. افتح Notebook Settings ثم Accelerator واختر GPU، وبعدها أعد تشغيل الخلية.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


In [ ]:
from pathlib import Path
import subprocess
repo_dir = Path('/kaggle/working/Myownai')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Eslamrady42/Myownai.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'fastapi', 'uvicorn', 'pydantic', 'psutil'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
print('تم تجهيز كود Myownai في:', repo_dir)


## إنشاء بيانات اختبار مؤقتة

هذه الجمل المصطنعة تتيح لنا اختبار خط التدريب كاملًا فقط. لا تستخدمها لتقييم جودة اللغة أو اعتبار النموذج مدرّبًا.


In [ ]:
from pathlib import Path
corpus_dir = repo_dir / 'datasets' / 'kaggle_smoke_test'
corpus_dir.mkdir(parents=True, exist_ok=True)
seed_texts = [
    "السلام عليكم، هذا اختبار تقني لنظام التدريب وحفظ النموذج.",
    "نحن نبني نموذج لغة مستقلًا ونتحقق من عمل خطوات التدريب.",
    "هذا النص تجريبي ولا يمثل بيانات لغوية حقيقية أو معرفة عامة.",
    "The purpose of this sentence is to test the training pipeline.",
    "We are checking tokenization, model training, checkpoints and generation.",
    "This synthetic corpus is only for a software smoke test.",
    "The model starts with random weights and has not learned language yet.",
    "يجب استخدام بيانات نملك حق استخدامها عند بدء التدريب الحقيقي.",
]
text = '\n'.join(f'{line} رقم العينة {i} Sample number {i}.' for i in range(1200) for line in seed_texts)
corpus_file = corpus_dir / 'smoke_test.txt'
corpus_file.write_text(text, encoding='utf-8')
print('حجم ملف الاختبار بالميجابايت:', round(corpus_file.stat().st_size / 1024**2, 2))


In [ ]:
from myownai.tokenizer import OwnTokenizer
artifact_dir = Path('/kaggle/working/myownai_smoke_test')
artifact_dir.mkdir(parents=True, exist_ok=True)
tokenizer_path = OwnTokenizer.train([corpus_file], artifact_dir / 'tokenizer', vocab_size=512)
tokenizer = OwnTokenizer(tokenizer_path)
print('Tokenizer path:', tokenizer_path)
print('Actual vocabulary size:', tokenizer.vocab_size)


In [ ]:
from myownai.config import ModelConfig
from myownai.data import encode_corpus
from myownai.training import train_model
config = ModelConfig(
    vocab_size=tokenizer.vocab_size,
    d_model=64,
    n_layers=2,
    n_heads=4,
    max_seq_len=64,
    dropout=0.05,
)
token_ids = encode_corpus(corpus_dir, tokenizer)
print('Token count:', token_ids.numel())
report = train_model(
    model_config=config,
    token_ids=token_ids,
    tokenizer_path=tokenizer_path,
    output_dir=artifact_dir,
    max_steps=20,
    batch_size=8,
    grad_accum_steps=1,
    eval_interval=5,
    validation_fraction=0.1,
)
print('Training device:', report['device'])
print('Parameters:', f"{report['parameter_count']:,}")
print('Steps:', report['steps'])
print('Checkpoint:', report['checkpoint'])
print('Final training loss:', report['history'][-1]['train_loss'])


In [ ]:
from myownai.inference import LocalGenerator
generator = LocalGenerator(artifact_dir / 'checkpoint.pt', artifact_dir / 'tokenizer.model')
print('عينة توليد تجريبية (قد تكون غير مفهومة وهذا طبيعي):')
print(generator.generate('السلام عليكم', max_new_tokens=40))


In [ ]:
import shutil
zip_path = shutil.make_archive('/kaggle/working/myownai_smoke_test', 'zip', artifact_dir)
print('تم تجهيز ملف النتائج:', zip_path)
print('بعد انتهاء التشغيل، افتح قسم Output في Kaggle لتنزيل myownai_smoke_test.zip.')
